# Verification of the MICADO Calibration Assembly (MCA)

This notebook describes how to simulate calibration exposures using the MICADO Calibration Assembly with ScopeSim. For this purpose a new submode `CALIB` has been introduced, which replaces the `SCAO` and `MCAO` submodes that are used for on-sky observations. It can be combined with all instrument modes, i.e. `IMG_4mas`, `SPEC`, etc. 
`CALIB` does not include the Armazones (atmosphere) and ELT effects. It currently includes the following to describe the MCA:
- `mca_mirror`: a single deployable mirror that is unique to the MCA. As with all mirrors the effect describes throughput (reflectivity) as well as thermal emission.
- `relay_surface_list`: This is the list of mirrors in the relay optics that is used in stand-alone mode, identical to the mirror list in the `SCAO` submode. Note that MORFEO is not yet supported for MCA simulations.
- `air_transmission`: The optical path from the MCA to the entrance window of MICADO has a length of about 14 metres through air, which therefore imprints an absorption signal on the input (continuum) spectrum. The effect uses a library of transmission spectra for various values of relative humidity (see below for details).
- `psf`: Very simplistically, the instrumental PSF (imprinted on observations using a pinhole mask) is modeled as Gaussian PSF of FWHM = 0.02 arcsec. This can be made more realistic in the future.

In [ ]:
import scopesim as sim

In [ ]:
sim.link_irdb("../../../")

If you have not done so already, please download the relevant instrument packages using the following code in a new cell:

```sim.download_packages(["MICADO"])```

Alternatively, if you would like to keep the instrument packages in a separate directory, you can set the following config value:

```sim.set_inst_pkgs_path("path/to/packages")```

In [ ]:
# sim.set_inst_pkgs_path("/Users/user/path/inst_pkgs")

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from astropy import units as u
from astropy.wcs import WCS

## Setting up the optical train
We first set up MICADO for the nominal imaging mode, and fix the "telescope area" to the MCA deployable mirror area:

In [ ]:
cmd = sim.UserCommands(use_instrument="MICADO", set_modes=["CALIB", "IMG_4mas"])

Uncomment to include debug messages

In [ ]:
# from scopesim import set_console_log_level
# sim.set_console_log_level("DEBUG")
# from scopesim.optics.image_plane_utils import logger
# logger.setLevel("DEBUG")

Instantiate the optical train and confirm the TEL.area

In [ ]:
micado = sim.OpticalTrain(cmd)
micado["pupil_wheel"].change_filter("open") # ND1
micado["filter_wheel_2"].change_filter("open") # Ks
micado["filter_wheel_1"].change_filter("Spec_IJ") # Spec_IJ
micado['slit_wheel'].change_slit("Short")
# micado['micado_wide_field_mirror_list'].include = False
# micado['air_transmission'].include = False
micado.cmds["!TEL.area"]

### Check mirror areas
A snippet to print all the mirror areas in the effects list

In [ ]:
# micado.cmds["!TEL"]
# from scopesim.effects import SurfaceList
# list = micado.optics_manager.get_all(SurfaceList)
# for s in list:
#     print(s.display_name, s.area)

In [ ]:
micado.effects.pprint_all()

## Imaging 
The MICADO calibration mode needs to use a `Source` object (unlike the METIS WCU mode). We use a flat field from `Scopesim_Targets`.

First, we define the pixel scale of the *source* and its angular extent in *mas*

In [ ]:
from synphot import SourceSpectrum, units
from synphot.models import Empirical1D
from scopesim_targets.extended_source import Flat
from astropy.table import QTable, unique
from astropy.io import fits

PIXSCALE = 64.0 # pixel scale in mas
NX = int(1024 * 4 // PIXSCALE) # size of flat lamp in pixels
NY = int(1024 * 4 // PIXSCALE) # size of flat lamp in pixels

Read flux definition of the flat lamp from file...

In [ ]:
# COLUMN = "9V"
# ROOT = "../../../../flatcube/"

# tbl = QTable.read(ROOT+"rad.ecsv")
# wave = tbl["wavelength"]
# flux = tbl[COLUMN]
# flux

...or define it as a black-body spectrum

In [ ]:
from astropy.modeling.physical_models import BlackBody
from astropy import constants as c

bb = BlackBody(temperature = 2000*u.K)
wave = np.linspace(0.7, 2.5, 10000)*u.um
wave <<= u.nm
energy_flux_nu = bb(wave)

photon_energy = (c.h * c.c / wave).to(u.erg)

# Photon flux per Hz: (Energy Flux) / (Energy per Photon)
# Units: photon / (cm2 s Hz sr)
photon_flux_nu = (energy_flux_nu / photon_energy) * u.photon
photon_flux_lam = photon_flux_nu.to(
    u.photon / (u.cm**2 * u.s * u.AA * u.sr), 
    equivalencies=u.spectral_density(wave)
)
# arbitrary scaling factor
flux = photon_flux_lam * 1e-5 # * (1*u.arcsec**2)
flux <<= u.Unit("ph / (nm s sr cm2)")
flux 

### 2D source definition

In [ ]:
spec = SourceSpectrum(Empirical1D, points=wave, lookup_table=flux * (1*u.arcsec**2))

ref_wave = 1.75*u.um
ref_flux = spec(ref_wave).to(u.Jy, u.spectral_density(ref_wave))

tg = Flat(spectrum=spec, brightness=(ref_wave, ref_flux/u.arcsec**2))

grid = {
    "pixel_scale": (PIXSCALE * u.mas/u.pix) << u.arcsec/u.pix,
    "width": NX,
    "height": NY,
}

flat_2d = tg.to_source(grid)

In [ ]:
# flux at reference wavelength
spec(ref_wave)
# ref value: 2.5045721 photlam

### Cube source definition

Re-read the table from file, or comment out to use the previous flux definition

In [ ]:
dwave = np.diff(wave)
delta_wave = dwave[0]
n_wave = len(wave)
pixel_scale = PIXSCALE * u.mas

cube = np.broadcast_to(
    flux.value.astype("float32")[:, None, None], (n_wave, NY, NX)
).copy()

w = WCS(naxis=3)
w.wcs.ctype = ["LINEAR", "LINEAR", "WAVE"]
w.wcs.cunit = ["deg", "deg", "nm"]
w.wcs.crpix = [NX / 2 + 0.5, NY / 2 + 0.5, 1]
w.wcs.crval = [0.0, 0.0, wave[0].value]
pix_scale_deg = pixel_scale.to(u.deg).value
w.wcs.cdelt = [pix_scale_deg, pix_scale_deg, delta_wave.value]

header = w.to_header()
header["BUNIT"] = flux.unit.to_string()
print(f"{header["BUNIT"] = }")

hdu = fits.PrimaryHDU(data=cube, header=header)
flat_cube = sim.Source(cube=hdu)

print(f"delta_wave: {delta_wave}")
print(f"pixel_area: {pixel_scale**2}")
print(f"field_area: {NX*NY*pixel_scale**2 << u.arcsec**2}")


## Back to the simulation

Plot the current filter's transmission

In [ ]:
micado["filter_wheel_1"].current_filter.plot();

In [ ]:
# Observe either the 2d or cube flat lamp source
micado.observe(flat_2d)

# Reference values for the BB spectrum, Spec_IJ filter, short slit

# 2D flat @ 64 mas pixels
# astar.scopesim.optics.fov - 2D FOV make_hdu: canvas_image_hdu.data.mean() = 1146701.811449 ph/s/pix

# 2D flat @ 32 mas pixels
# astar.scopesim.optics.fov - 2D FOV make_hdu: canvas_image_hdu.data.mean() = 1146701.811449 ph/s/pix

# 3D flat @ 64 mas pixels
# astar.scopesim.optics.fov - 2D FOV make_hdu: canvas_image_hdu.data.mean() = 1147022.169664 ph/s/pix

# 3D flat @ 32 mas pixels
# astar.scopesim.optics.fov - 2D FOV make_hdu: canvas_image_hdu.data.mean() = 1147022.169664 ph/s/pix

# Ks, short slit, no air, no wide-field-surf-list:
# astar.scopesim.optics.fov - 2D FOV make_hdu: canvas_image_hdu.data.mean() = 888745.590001 ph/s/pix

# MCA source, Spec_IJ, short slit:
# astar.scopesim.optics.fov - 2D FOV make_hdu: canvas_image_hdu.data.mean() = 655077.538545 ph/s/pix

### Flat souces comparison

First - inspect the properties of the 2D collapsed image

In [ ]:
from scopesim.source.source_fields import (
    HDUSourceField,
    ImageSourceField,
    CubeSourceField,
)

def flux_image(field: HDUSourceField):
    """Return 2D image in ph/s/cm2.

    Spectrum is integrated over whole range.
    Cubes are flattened and spectrally integrated.

    # usage e.g.:
    flximg = flux_image(micado._last_source.fields[0])
    """
    if isinstance(field, ImageSourceField):
        return field.data * field.spectrum.integrate()
    if isinstance(field, CubeSourceField):
        dlam = field.header["CDELT3"] * u.Unit(field.header["CUNIT3"])
        print(f"{field.bunit = }, {dlam = }, field.pixel_area = {field.pixel_area << u.mas**2 }")
        img = field.data.sum(axis=0) * field.bunit * dlam
        if field.is_bunit_spatially_differential:
            img *= field.pixel_area
        return img.to(u.ph/u.s/u.cm**2)
    raise TypeError(f"{type(field)} unsupported")

In [ ]:
# inspect either the source object or the source as ingested into the optical train

flximg = flux_image(flat_2d.fields[0])
print(f"flat_2d img shape {flximg.shape}")
print(f"flat_2d img mean {flximg.mean()}")

flximg = flux_image(flat_cube.fields[0])
print(f"flat_cube img shape {flximg.shape}")
print(f"flat_cube img mean {flximg.mean()}")

# flximg = flux_image(micado._last_source.fields[0])

# Referene values
# flat_2d: 142.18657012921608 ph / (s cm2) 
# flat_cube: 142.19518660314446 ph / (s cm2) 


In [ ]:
# check the header of the ingested source
# micado._last_source.fields[0].header

### Compare the spectra of the 2D and cube flat sources

In [ ]:
plt.figure(figsize=(6,3))
cube_field = flat_cube.fields[0]
flux = cube_field.data.mean(axis=(1,2)) * cube_field.bunit * cube_field.pixel_area << u.Unit("photlam")
print(f"{cube_field.bunit = }")
waveset = cube_field.waveset << u.Angstrom
spec = SourceSpectrum(Empirical1D, points=waveset, lookup_table=flux)
plt.plot(waveset, spec(waveset), label = "cube")

waveset_2d = flat_2d.fields[0].spectrum.waveset << u.Angstrom
plt.plot(waveset_2d, flat_2d.fields[0].spectrum(waveset_2d) / (NX*NY), label="2d")
plt.xlabel("wavelength [A]")
plt.ylabel("flux [ph / (s cm^2 A)]")
plt.legend();

### Image plane check

In [ ]:
im = micado.image_planes[0].data
print("Sum:     ", im.sum() * u.Unit("ph/s"))
print("Max:     ", im.max() * u.Unit("ph/s/pix"))
print("Mean:     ", im.mean() * u.Unit("ph/s/pix"))
# plt.imshow(im)
# plt.colorbar();

# Reference values from BB source, Spec_IJ filter, Short slit
# 2D_Flat @ 64 mas source pixel size
# Sum:      3440105437.5108867 ph / s
# Max:      1146701.8125036291 ph / (pix s)

# 2D @ 32
# Sum:      3440105437.510886 ph / s
# Max:      1146701.812503629 ph / (pix s)

# 3D @ 64
# Sum:      3441064137.2765436 ph / s
# Max:      1147021.3790921816 ph / (pix s)

# 3D @ 32
# Sum:      3441064137.276469 ph / s
# Max:      1147021.379092156 ph / (pix s)

# Reference values from radiometry file
# 5V 2D @ 64, Ks, open - Mean:      87709.80445358888 ph / (pix s)
# 5V 2D @ 64, Ks, ND1  - Mean:      8770.980445358895 ph / (pix s)
# 5V 2D @ 64, Ks, ND3  - Mean:      87.70980445358875 ph / (pix s)

# Sum refernce values
# Sum:      2712488860.1544886 ph / s (BB, Ks, no air, no wide-img-surfaces)
#           1964701785.480649 (MCA, Spec_IJ, 2d_source)
#           1965249571.123346 (MCA, Spec_IJ, 3d_source)
#           2705219519 (spec total, BB, Ks, no air, no grating_eff)

In [ ]:
micado.image_planes[0].hdu.writeto("./out/img_implane.fits", overwrite=True)

In [ ]:
plt.imshow(im)
plt.colorbar();

### Readout check

In [ ]:
readout = micado.readout(dit=1, ndit=1)[0]

In [ ]:
print("Sum:     ", readout[1].data.sum())
print("Max:     ", readout[1].data.max())
print("Mean:     ", readout[1].data.mean())
print("Std. dev.:", readout[1].data.std())

In [ ]:
mask = readout[1].data > 100
plt.hist(readout[1].data[mask], bins=100);

# Spectroscopy

In [ ]:
cmd = sim.UserCommands(use_instrument="MICADO", set_modes=["CALIB", "SPEC"])
micado = sim.OpticalTrain(cmd)
cmd["!TEL.area"]

We will observe the flat lamp, which allows us to switch off the psf effects. We'll try to simulate a full field of view.

In [ ]:
micado['psf'].include = False
micado['micado_ncpas_psf'].include = False
micado['filter_wheel_1'].change_filter("Spec_IJ")
micado['filter_wheel_2'].change_filter("open")
micado['detector_window'].include = False
micado['full_detector_array'].include = True
micado['grating_efficiency'].include = True
micado['air_transmission'].include = True

In [ ]:
micado.effects.pprint_all()

The transmission of the 14 meter air column in the MCA and relay optics is provided by the `air_transmission` effect. This is a library of transmission spectra for relative humidities between 5 and 95 per cent, available in steps of 5 per cent. The default is 10 per cent, which can be changed with the `update()` method:

In [ ]:
air = micado['air_transmission']

In [ ]:
air = micado['air_transmission']
print("Default humidity:", air.meta['relH'], "(per cent)")

In [ ]:
air.update(relH=0.21)
print("Current humidity:", air.meta['relH'], "(per cent)")

In [ ]:
waveset = air.throughput.waveset
t_air = air.throughput
plt.figure(figsize=(6,2))
plt.plot(waveset, t_air(waveset))
plt.xlabel("Wavelength [A]")
plt.ylabel("Throughput [normalised]");

In [ ]:
micado.observe(flat_2d)

# Reference values with BB, Spec_IJ, short slit, air with RH=21%
# 2d @ 32, FOV1, ORDER_3_1:
# astar.scopesim.optics.fov - 3D FOV make_imagefields: field_cube.mean() = <Quantity 0.55063668 PHOTLAM>
# astar.scopesim.optics.fov - 3D FOV make_hdu: canvas_cube_hdu.data.mean() = 53700005148.001999

# 2d @ 64, FOV1:
# astar.scopesim.optics.fov - 3D FOV make_imagefields: field_cube.mean() = <Quantity 0.55063668 PHOTLAM>
# astar.scopesim.optics.fov - 3D FOV make_hdu: canvas_cube_hdu.data.mean() = 53700005148.001999

# 3d @ 32, FOV1:
# astar.scopesim.optics.fov - 3D FOV make_cubefields: field_data.mean() = 0.550609 [PHOTLAM / arcsec2]
# astar.scopesim.optics.fov - 3D FOV make_hdu: canvas_cube_hdu.data.mean() = 53697302115.919312

# 3d @ 64, FOV1:
# astar.scopesim.optics.fov - 3D FOV make_cubefields: field_data.mean() = 0.550609 [PHOTLAM / arcsec2]
# astar.scopesim.optics.fov - 3D FOV make_hdu: canvas_cube_hdu.data.mean() = 53697302115.921593


In [ ]:
hdu = micado.fov_manager.fovs[0].view()

In [ ]:
hdu.writeto("./out/spec_cube.fits", overwrite=True)

In [ ]:
print(f"{micado.image_planes[0].data.max() = }")

# MCA flux, Spec_IJ, short-slit
# max:
# 3d 5V @64: 
# 2d 9V @64: 133.33916769096788

# BB, Ks, short-slit, no air, no grating_eff
# sum (for spec total flux)
# 2d #64: 2705219519.086759


In [ ]:
micado.image_planes[0].hdu.writeto("./out/spec_implane.fits", overwrite=True)

In [ ]:
readout = micado.readout(dit=60, ndit=1, filename="./out/spec_readout.fits")[0]

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(readout[5].data)
# plt.imshow(micado.image_planes[0].data)
plt.colorbar();

In [ ]:
rect = micado['micado_spectral_traces'].rectify_traces(readout, -1.5, 1.5)

In [ ]:
print(f"{len(rect) = }")
print(f"{rect[3].header["EXTNAME"]}")

In [ ]:
plt.figure(figsize=(12,2))
plt.imshow(rect[3].data); 
plt.xlim(0, rect[3].data.shape[1]);
plt.xlabel("pixel");

In [ ]:
plt.figure(figsize=(15,2))
j = np.arange(rect[3].data.shape[1])
wcs = WCS(rect[3].header).spectral
lam = wcs.all_pix2world(j, 0)[0]
lam = (lam * wcs.wcs.cunit[0]).to(u.um)
L = spec(lam) / spec(lam).max()
plt.plot(lam, rect[3].data[300:500, ].mean(axis=0), label="spec_flat")
plt.plot(lam, np.divide(rect[3].data[300:500, ].mean(axis=0), L), label="lamp-normalised response")
# mask = air.throughput(lam) > 0.8
# data = rect[3].data[:, mask] / L[mask]
# plt.plot(lam[mask], np.divide(data[300:301,].mean(axis=0), air.throughput(lam)[mask]), label="normalised response")
plt.title(rect[3].header["EXTNAME"])
plt.legend()
plt.xlabel("Wavelength [um]");